# GroupBy and Aggregation using PySpark

In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Practise').getOrCreate()

In [4]:
df = spark.read.csv("../datasets/sample_data_groupby.csv", header = True, inferSchema=True)

In [5]:
df.show()

+---------+------------+------+
|     Name| Departments|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
+---------+------------+------+



In [6]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- salary: integer (nullable = true)



## Groupby

- an aggregation function is generally used after groupby()

In [10]:
df.groupby('Name').sum().show()

+---------+-----------+
|     Name|sum(salary)|
+---------+-----------+
|Sudhanshu|      35000|
|    Sunny|      10000|
|    Krish|      19000|
|   Mahesh|       7000|
+---------+-----------+



- which department has the maximum salary

In [12]:
df.groupby('Departments').sum().show()

+------------+-----------+
| Departments|sum(salary)|
+------------+-----------+
|         IOT|      15000|
|    Big Data|      13000|
|Data Science|      43000|
+------------+-----------+



In [13]:
df.groupby('Departments').mean().show()

+------------+-----------------+
| Departments|      avg(salary)|
+------------+-----------------+
|         IOT|           7500.0|
|    Big Data|4333.333333333333|
|Data Science|          10750.0|
+------------+-----------------+



- how many employees are working in each department

In [15]:
df.groupby('Departments').count().show()

+------------+-----+
| Departments|count|
+------------+-----+
|         IOT|    2|
|    Big Data|    3|
|Data Science|    4|
+------------+-----+



- what is the total salary given across all departments

In [17]:
df.agg({'Salary':'sum'}).show()

+-----------+
|sum(Salary)|
+-----------+
|      71000|
+-----------+



# Using transaction dataset

In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, count, sum, avg, mean, min, max,
    stddev, countDistinct, round,
    when, lit, desc, asc
)

In [2]:
spark = SparkSession.builder.appName('Practise').getOrCreate()

In [3]:
df = spark.read.csv('../datasets/transactions_v3.csv', header = True, inferSchema= True )

In [4]:
df.show(truncate=False)

+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------+-----------------+--------------------+
|transaction_id|customer_id|name       |age|account_type|balance |transaction_amount|transaction_type|city     |transaction_date|credit_score|branch           |relationship_manager|
+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------+-----------------+--------------------+
|T001          |C101       |Arun Sharma|34 |Savings     |45000.0 |5000.0            |Debit           |Mumbai   |2024-01-15      |720         |Mumbai-North     |RM01                |
|T002          |C102       |Priya Sen  |28 |Current     |120000.0|15000.0           |Credit          |Delhi    |2024-01-16      |680         |Delhi-East       |RM02                |
|T003          |C103       |Rahul Das  |45 |Savings     |8000.0  |2000.0            |Debit

### Total Transaction amount by city

In [6]:
df.groupby('city').sum('transaction_amount').show()

+---------+-----------------------+
|     city|sum(transaction_amount)|
+---------+-----------------------+
|Bangalore|                16000.0|
|  Chennai|                30000.0|
|   Mumbai|                14000.0|
|  Kolkata|                14000.0|
|     Pune|                 3000.0|
|    Delhi|                23000.0|
|Hyderabad|                 7500.0|
+---------+-----------------------+



In [7]:
# Average credit score by account type
df.groupBy("account_type").avg("credit_score").show()


+------------+-----------------+
|account_type|avg(credit_score)|
+------------+-----------------+
|        Loan|            610.0|
|     Savings|681.1111111111111|
|     Current|            695.0|
+------------+-----------------+



In [8]:
# Count of transactions by transaction type
df.groupBy("transaction_type").count().show()

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|          Credit|    6|
|           Debit|    9|
+----------------+-----+



In [9]:
# Multiple aggregations using a dictionary
df.groupBy("city").agg({
    "transaction_amount": "sum",
    "credit_score": "avg",
    "transaction_id": "count"
}).show()

+---------+---------------------+-----------------+-----------------------+
|     city|count(transaction_id)|avg(credit_score)|sum(transaction_amount)|
+---------+---------------------+-----------------+-----------------------+
|Bangalore|                    2|            755.0|                16000.0|
|  Chennai|                    2|            610.0|                30000.0|
|   Mumbai|                    3|            720.0|                14000.0|
|  Kolkata|                    2|            590.0|                14000.0|
|     Pune|                    2|            640.0|                 3000.0|
|    Delhi|                    2|            680.0|                23000.0|
|Hyderabad|                    2|            710.0|                 7500.0|
+---------+---------------------+-----------------+-----------------------+



### In production we use function styled agg

In [11]:
df.groupBy("city").agg(
    count("transaction_id").alias("num_transactions"),
    sum("transaction_amount").alias("total_volume"),
    avg("transaction_amount").alias("avg_txn_amount"),
    min("transaction_amount").alias("min_txn_amount"),
    max("transaction_amount").alias("max_txn_amount"),
    round(avg("credit_score"), 0).alias("avg_credit_score")
).show()

+---------+----------------+------------+-----------------+--------------+--------------+----------------+
|     city|num_transactions|total_volume|   avg_txn_amount|min_txn_amount|max_txn_amount|avg_credit_score|
+---------+----------------+------------+-----------------+--------------+--------------+----------------+
|Bangalore|               2|     16000.0|           8000.0|        7000.0|        9000.0|           755.0|
|  Chennai|               2|     30000.0|          15000.0|        5000.0|       25000.0|           610.0|
|   Mumbai|               3|     14000.0|4666.666666666667|        3000.0|        6000.0|           720.0|
|  Kolkata|               2|     14000.0|           7000.0|        2000.0|       12000.0|           590.0|
|     Pune|               2|      3000.0|           1500.0|        1000.0|        2000.0|           640.0|
|    Delhi|               2|     23000.0|          11500.0|        8000.0|       15000.0|           680.0|
|Hyderabad|               2|      750

## Groupby on multiple columns

In [15]:
df.groupBy("city", "transaction_type").agg(
    count("*").alias("num_transactions"),
    sum("transaction_amount").alias("total_amount")
).orderBy("city", "transaction_type").show()

+---------+----------------+----------------+------------+
|     city|transaction_type|num_transactions|total_amount|
+---------+----------------+----------------+------------+
|Bangalore|          Credit|               1|      7000.0|
|Bangalore|           Debit|               1|      9000.0|
|  Chennai|          Credit|               1|     25000.0|
|  Chennai|           Debit|               1|      5000.0|
|    Delhi|          Credit|               1|     15000.0|
|    Delhi|           Debit|               1|      8000.0|
|Hyderabad|          Credit|               1|      4500.0|
|Hyderabad|           Debit|               1|      3000.0|
|  Kolkata|           Debit|               2|     14000.0|
|   Mumbai|          Credit|               1|      6000.0|
|   Mumbai|           Debit|               2|      8000.0|
|     Pune|          Credit|               1|      2000.0|
|     Pune|           Debit|               1|      1000.0|
+---------+----------------+----------------+-----------

### Sorting aggregated results

In [17]:
from pyspark.sql.functions import desc, asc

In [18]:
df.groupBy("city").agg(
    sum("transaction_amount").alias("total_volume")
).orderBy(desc("total_volume")).show()

+---------+------------+
|     city|total_volume|
+---------+------------+
|  Chennai|     30000.0|
|    Delhi|     23000.0|
|Bangalore|     16000.0|
|   Mumbai|     14000.0|
|  Kolkata|     14000.0|
|Hyderabad|      7500.0|
|     Pune|      3000.0|
+---------+------------+



In [20]:
# Alternatively using col()
df.groupBy("city").agg(
    sum("transaction_amount").alias("total_volume")
).orderBy(col("total_volume").desc()).show()


+---------+------------+
|     city|total_volume|
+---------+------------+
|  Chennai|     30000.0|
|    Delhi|     23000.0|
|Bangalore|     16000.0|
|   Mumbai|     14000.0|
|  Kolkata|     14000.0|
|Hyderabad|      7500.0|
|     Pune|      3000.0|
+---------+------------+



### Unique value count

In [22]:
from pyspark.sql.functions import countDistinct

In [23]:
df.groupBy("city").agg(
    countDistinct("customer_id").alias("unique_customers"),
    count("*").alias("total_transactions")
).orderBy(desc("unique_customers")).show()

+---------+----------------+------------------+
|     city|unique_customers|total_transactions|
+---------+----------------+------------------+
|Bangalore|               1|                 2|
|  Chennai|               1|                 2|
|   Mumbai|               1|                 3|
|  Kolkata|               1|                 2|
|     Pune|               1|                 2|
|    Delhi|               1|                 2|
|Hyderabad|               1|                 2|
+---------+----------------+------------------+



In [24]:
df.groupBy("branch").agg(
    countDistinct("customer_id").alias("unique_customers"),
    count("transaction_id").alias("total_transactions"),
    round(
        count("transaction_id") / countDistinct("customer_id"), 2
    ).alias("avg_txns_per_customer")
).show()

+-----------------+----------------+------------------+---------------------+
|           branch|unique_customers|total_transactions|avg_txns_per_customer|
+-----------------+----------------+------------------+---------------------+
|  Bangalore-South|               1|                 2|                  2.0|
|       Delhi-East|               1|                 2|                  2.0|
|        Pune-West|               1|                 2|                  2.0|
|Hyderabad-Central|               1|                 2|                  2.0|
|  Kolkata-Central|               1|                 2|                  2.0|
|     Mumbai-North|               1|                 3|                  3.0|
|    Chennai-South|               1|                 2|                  2.0|
+-----------------+----------------+------------------+---------------------+



### Aggregation without groupby

In [26]:
df.agg(
    count("*").alias("total_transactions"),
    countDistinct("customer_id").alias("unique_customers"),
    sum("transaction_amount").alias("total_volume"),
    round(avg("transaction_amount"), 2).alias("avg_txn_amount"),
    round(avg("credit_score"), 1).alias("avg_credit_score"),
    min("transaction_date").alias("earliest_txn"),
    max("transaction_date").alias("latest_txn")
).show(truncate=False)

+------------------+----------------+------------+--------------+----------------+------------+----------+
|total_transactions|unique_customers|total_volume|avg_txn_amount|avg_credit_score|earliest_txn|latest_txn|
+------------------+----------------+------------+--------------+----------------+------------+----------+
|15                |7               |107500.0    |7166.67       |675.3           |2024-01-15  |2024-01-29|
+------------------+----------------+------------+--------------+----------------+------------+----------+



### Filtering after groupby

In [28]:
# Only show cities with more than 2 transactions
df.groupBy("city").agg(
    count("*").alias("num_transactions"),
    sum("transaction_amount").alias("total_volume")
).filter(col("num_transactions") > 2) \
 .orderBy(desc("total_volume")) \
 .show()

+------+----------------+------------+
|  city|num_transactions|total_volume|
+------+----------------+------------+
|Mumbai|               3|     14000.0|
+------+----------------+------------+

